## Preprocessing

In [13]:
import json
import pandas as pd

# 1. Caricamento del dataset grezzo
input_path = '../data/raw/MC3_graph.json'
with open(input_path, 'r', encoding='utf-8') as f:
    graph_data = json.load(f)

nodes = graph_data['nodes']
edges = graph_data['edges']

print("==================================================")
print("  ESTRAZIONE AUTOMATICA DELLE DIMENSIONI (PUNTO 2.2) ")
print("==================================================\n")

# --- DIMENSIONE 1: T NODES (Mappatura dei Tipi di Nodo) ---
df_nodes = pd.DataFrame(nodes)
print("--- [T NODES] Sotto-tipi di Nodi presenti nel Grafo ---")
t_nodes_summary = df_nodes.groupby(['type', 'sub_type']).size().reset_index(name='Quantità')
print(t_nodes_summary.to_string(index=False))
print("\n" + "-"*50 + "\n")


# --- DIMENSIONE 2: T LINKS (Mappatura dei Tipi di Relazione) ---
df_edges = pd.DataFrame(edges)
df_edges['type_mapped'] = df_edges['type'].fillna('structural_link (Collegamento Entità-Relazione)')
print("--- [T LINKS] Tipi di Collegamenti espliciti negli Archi ---")
t_links_summary = df_edges.groupby('type_mapped').size().reset_index(name='Quantità')
print(t_links_summary.to_string(index=False))
print("\n" + "-"*50 + "\n")


# --- DIMENSIONE 3: P NODES (Proprietà e Identità Verificate vs Sospette) ---
print("--- [P NODES] Proprietà disponibili per Sotto-tipo di Nodo ---")
for (macro_type, sub_type), group in df_nodes.groupby(['type', 'sub_type']):
    # Troviamo solo le colonne che non sono interamente nulle per quel sottotipo
    active_props = group.dropna(how='all', axis=1).columns.tolist()
    print(f"• {macro_type} -> {sub_type}: {active_props}")

# Analisi Identità (Verificata vs Sospetta/Pseudonimo) per le persone
print("\n* Focus Investigativo P Nodes (Identità Person):")
df_persons = df_nodes[df_nodes['sub_type'] == 'Person']
# Identifichiamo gli pseudonimi (nomi brevi o descrittivi che non sono nomi propri completi)
pseudonyms = df_persons[df_persons['name'].str.contains('Boss|Lookout|Accountant|Fry|Operator|Driver|Guest', case=False, na=False)]
print(f"  - Totale Persone mappate: {len(df_persons)}")
print(f"  - Di cui Pseudonimi/Sospetti rilevati nel dataset: {pseudonyms['name'].tolist()}")
print("\n" + "-"*50 + "\n")


# --- DIMENSIONE 4: P LINKS (Proprietà dei Legami e Livello di Certezza) ---
print("--- [P LINKS] Proprietà e Livello di Certezza dei Legami ---")
p_links_props = [col for col in df_edges.columns if col != 'type_mapped']
print(f"• Attributi nativi degli archi: {p_links_props}")

# Analisi del Livello di Certezza (is_inferred)
certezza = df_edges['is_inferred'].fillna(False).value_counts()
print(f"\n* Analisi di Certezza dei Legami (is_inferred):")
print(f"  - Legami CERTI (Osservati direttamente): {certezza.get(False, 0)}")
print(f"  - Legami INFERTI/SOSPETTI (Dedotti dall'investigatore): {certezza.get(True, 0)}")

  ESTRAZIONE AUTOMATICA DELLE DIMENSIONI (PUNTO 2.2) 

--- [T NODES] Sotto-tipi di Nodi presenti nel Grafo ---
        type         sub_type  Quantità
      Entity            Group         5
      Entity         Location        29
      Entity     Organization         5
      Entity           Person        18
      Entity           Vessel        15
       Event       Assessment        36
       Event      Collaborate        25
       Event    Communication       584
       Event        Criticize         1
       Event      Enforcement        21
       Event          Fishing         1
       Event     HarborReport         2
       Event       Monitoring        70
       Event     TourActivity        13
       Event  TransponderPing         3
       Event   VesselMovement        46
Relationship AccessPermission        68
Relationship       Colleagues        30
Relationship      Coordinates        74
Relationship          Friends         2
Relationship     Jurisdiction        13
Relations

In [14]:
import json
import pandas as pd
import os

# Percorsi dei file
input_path = '../data/raw/MC3_graph.json'
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

# 1. Caricamento del Grafo Completo
print("1. Caricamento del Knowledge Graph in corso...")
with open(input_path, 'r', encoding='utf-8') as f:
    vast_data = json.load(f)

# Estrazione delle liste grezze
nodes_raw = vast_data['nodes']
edges_raw = vast_data['edges']
print(f"-> Il grafo contiene {len(nodes_raw)} nodi e {len(edges_raw)} archi.\n")

# 2. PROFILING ACCURATO DEI NODI
# Poiché i nodi hanno attributi diversi a seconda del tipo, creiamo dei DataFrame separati
print("2. Profiling e separazione dei nodi per macro-tipo...")
df_all_nodes = pd.DataFrame(nodes_raw)

# Vediamo quali colonne effettive sono state estratte (comprese quelle dinamiche)
print(f"Tutti gli attributi totali trovati nei nodi: {list(df_all_nodes.columns)}")

# Separiamo per 'type' (Entity, Event, Relationship) per non fare confusione con i valori nulli
df_entities = df_all_nodes[df_all_nodes['type'] == 'Entity'].dropna(how='all', axis=1)
df_events = df_all_nodes[df_all_nodes['type'] == 'Event'].dropna(how='all', axis=1)
df_relationships = df_all_nodes[df_all_nodes['type'] == 'Relationship'].dropna(how='all', axis=1)

print(f"   - Nodi 'Entity' (Persone, Barche, Luoghi): {len(df_entities)} | Attributi: {list(df_entities.columns)}")
print(f"   - Nodi 'Event' (Comunicazioni, Spostamenti): {len(df_events)} | Attributi: {list(df_events.columns)}")
print(f"   - Nodi 'Relationship' (Collegamenti logici): {len(df_relationships)} | Attributi: {list(df_relationships.columns)}\n")

# 3. PROFILING ACCURATO DEGLI ARCHI (EDGES)
print("3. Profiling e arricchimento degli archi...")
df_edges = pd.DataFrame(edges_raw)

# Gestione corretta dell'incertezza (is_inferred)
if 'is_inferred' in df_edges.columns:
    df_edges['is_inferred'] = df_edges['is_inferred'].fillna(False)
else:
    df_edges['is_inferred'] = False

print(f"Attributi totali trovati negli archi: {list(df_edges.columns)}")
# Contiamo quanti archi sono certi e quanti dedotti (Uncertainty)
inferred_count = df_edges['is_inferred'].sum()
print(f"   - Archi Certi (oggettivi): {len(df_edges) - inferred_count}")
print(f"   - Archi Inferred (ipotesi investigative di Clepper): {inferred_count}\n")

# 4. SALVATAGGIO DEI DATI PRESERVANDO TUTTI GLI ATTRIBUTI
# Salviamo in formato CSV separato ma completo, così la dashboard caricherà solo ciò che serve
print("4. Salvataggio dei dataset accurati in corso...")
df_entities.to_csv(f"{output_dir}/nodes_entities.csv", index=False)
df_events.to_csv(f"{output_dir}/nodes_events.csv", index=False)
df_relationships.to_csv(f"{output_dir}/nodes_relationships.csv", index=False)
df_edges.to_csv(f"{output_dir}/edges_complete.csv", index=False)

print("Operazione completata con successo! Nessun dato è andato perduto.")
df_nodes.to_csv(output_nodes, index=False)
df_edges.to_csv(output_edges, index=False)

print(f"Pulizia completata! File salvati in: \n- {output_nodes}\n- {output_edges}")

1. Caricamento del Knowledge Graph in corso...
-> Il grafo contiene 1159 nodi e 3226 archi.

2. Profiling e separazione dei nodi per macro-tipo...
Tutti gli attributi totali trovati nei nodi: ['type', 'label', 'name', 'sub_type', 'id', 'timestamp', 'monitoring_type', 'findings', 'content', 'assessment_type', 'results', 'movement_type', 'destination', 'enforcement_type', 'outcome', 'activity_type', 'participants', 'thing_collected', 'reference', 'date', 'time', 'friendship_type', 'permission_type', 'start_date', 'end_date', 'report_type', 'submission_date', 'jurisdiction_type', 'authority_level', 'coordination_type', 'operational_role']
   - Nodi 'Entity' (Persone, Barche, Luoghi): 72 | Attributi: ['type', 'label', 'name', 'sub_type', 'id']
   - Nodi 'Event' (Comunicazioni, Spostamenti): 802 | Attributi: ['type', 'label', 'sub_type', 'id', 'timestamp', 'monitoring_type', 'findings', 'content', 'assessment_type', 'results', 'movement_type', 'destination', 'enforcement_type', 'outcome', '

In [15]:
print("=== VALORI MANCANTI NEI NODI 'ENTITY' ===")
print(df_entities.isnull().sum())
print("\n" + "="*40 + "\n")

print("=== VALORI MANCANTI NEI NODI 'EVENT' ===")
# Vediamo anche la percentuale per capire l'impatto
missing_events = df_events.isnull().sum()
total_events = len(df_events)
for col, count in missing_events.items():
    pct = (count / total_events) * 100
    print(f"Colonna '{col}': {count} valori mancanti ({pct:.1f}%)")
    
print("\n" + "="*40 + "\n")

print("=== VALORI MANCANTI NEGLI ARCHI (EDGES) ===")
print(df_edges.isnull().sum())

=== VALORI MANCANTI NEI NODI 'ENTITY' ===
type        0
label       0
name        0
sub_type    0
id          0
dtype: int64


=== VALORI MANCANTI NEI NODI 'EVENT' ===
Colonna 'type': 0 valori mancanti (0.0%)
Colonna 'label': 0 valori mancanti (0.0%)
Colonna 'sub_type': 0 valori mancanti (0.0%)
Colonna 'id': 0 valori mancanti (0.0%)
Colonna 'timestamp': 32 valori mancanti (4.0%)
Colonna 'monitoring_type': 732 valori mancanti (91.3%)
Colonna 'findings': 732 valori mancanti (91.3%)
Colonna 'content': 218 valori mancanti (27.2%)
Colonna 'assessment_type': 769 valori mancanti (95.9%)
Colonna 'results': 770 valori mancanti (96.0%)
Colonna 'movement_type': 769 valori mancanti (95.9%)
Colonna 'destination': 761 valori mancanti (94.9%)
Colonna 'enforcement_type': 781 valori mancanti (97.4%)
Colonna 'outcome': 783 valori mancanti (97.6%)
Colonna 'activity_type': 798 valori mancanti (99.5%)
Colonna 'participants': 801 valori mancanti (99.9%)
Colonna 'thing_collected': 801 valori mancanti (99.9%)

In [16]:
import pandas as pd
import numpy as np

print("Inizio correzione e standardizzazione dei dati...")

# --- 1. SISTEMAZIONE NODI EVENTI ---
# Per gli attributi testuali o strutturali mancanti, mettiamo stringhe vuote o standard
# in modo che JavaScript non legga 'undefined' o 'NaN' rompendo la UI.
colonne_testuali_eventi = [
    'content', 'monitoring_type', 'findings', 'assessment_type', 'results', 
    'movement_type', 'destination', 'enforcement_type', 'outcome', 
    'activity_type', 'participants', 'thing_collected', 'reference', 'date', 'time'
]

for col in colonne_testuali_eventi:
    if col in df_events.columns:
        df_events[col] = df_events[col].fillna('')

# Gestione dei 32 Timestamp mancanti:
# Assegniamo un valore sentinella "Unknown_Time" o cerchiamo di non romperli.
# Nella dashboard li gestiremo mettendoli in una categoria "Senza Orario" per l'investigatore.
df_events['timestamp'] = df_events['timestamp'].fillna('Unknown_Time')


# --- 2. SISTEMAZIONE ARCHI (EDGES) ---
# Se manca l'ID dell'arco, creiamo una Serie di ID autogenerati e la usiamo nel fillna
generated_ids = pd.Series([f"generated_edge_{x}" for x in df_edges.index], index=df_edges.index)
df_edges['id'] = df_edges['id'].fillna(generated_ids)

# Se manca il tipo di arco (accade per 1022 archi), lo rinominiamo in 'structural_link'
df_edges['type'] = df_edges['type'].fillna('structural_link')


# --- 3. SALVATAGGIO DEFINITIVO ---
# Sovrascriviamo i file nella cartella processed con i dati perfettamente puliti e consistenti
df_entities.to_csv('../data/processed/nodes_entities.csv', index=False)
df_events.to_csv('../data/processed/nodes_events.csv', index=False)
df_relationships.to_csv('../data/processed/nodes_relationships.csv', index=False)
df_edges.to_csv('../data/processed/edges_complete.csv', index=False)

print("\n[SUCCESSO] I dati sono stati corretti, standardizzati e salvati!")
print(f"Verifica Archi - ID mancanti: {df_edges['id'].isnull().sum()} | Type mancanti: {df_edges['type'].isnull().sum()}")
print(f"Verifica Eventi - Timestamp mancanti: {df_events['timestamp'].isnull().sum()}")

Inizio correzione e standardizzazione dei dati...

[SUCCESSO] I dati sono stati corretti, standardizzati e salvati!
Verifica Archi - ID mancanti: 0 | Type mancanti: 0
Verifica Eventi - Timestamp mancanti: 0


In [17]:
import pandas as pd

print("Inizio Fase 2.3: Arricchimento dati per la Visual Analytics...")

# 1. Rileggiamo i file temporanei appena salvati
df_entities = pd.read_csv('../data/processed/nodes_entities.csv')
df_events = pd.read_csv('../data/processed/nodes_events.csv')
df_edges = pd.read_csv('../data/processed/edges_complete.csv')

# --- ENRICHMENT TEMPORALE (Sui nodi EVENT) ---
print("-> Estrazione delle feature temporali dagli eventi...")

# Convertiamo la colonna timestamp in un oggetto Datetime di Pandas
# errors='coerce' trasformerà i nostri 'Unknown_Time' in NaT (Not a Time), che gestiremo
df_events['datetime_parsed'] = pd.to_datetime(df_events['timestamp'], errors='coerce')

# Estraiamo l'ora (0-23), la data (YYYY-MM-DD) e il giorno della settimana
df_events['hour'] = df_events['datetime_parsed'].dt.hour.fillna(-1).astype(int)
df_events['date_only'] = df_events['datetime_parsed'].dt.strftime('%Y-%m-%d').fillna('Unknown')
df_events['day_of_week'] = df_events['datetime_parsed'].dt.day_name().fillna('Unknown')

# Rimuoviamo la colonna di parsing temporanea
df_events.drop(columns=['datetime_parsed'], inplace=True)


# --- ENRICHMENT TOPOLOGICO (Sui nodi ENTITY) ---
print("-> Calcolo della Degree Centrality per le entità...")

# Contiamo quante volte ogni nodo compare come 'source' o come 'target' negli archi
source_counts = df_edges['source'].value_counts()
target_counts = df_edges['target'].value_counts()

# Sommiamo i due conteggi per ottenere il grado totale di connessioni di ogni entità
total_degree = source_counts.add(target_counts, fill_value=0).astype(int)

# Mappiamo questa metrica nel DataFrame delle entità
df_entities['degree'] = df_entities['id'].map(total_degree).fillna(0).astype(int)


# --- SALVATAGGIO DEFINITIVO E FINALE ---
df_entities.to_csv('../data/processed/nodes_entities.csv', index=False)
df_events.to_csv('../data/processed/nodes_events.csv', index=False)

print("\n[SUCCESSO] Feature Engineering completata!")
print(f"Esempio Entità arricchite (Primi 3 nodi con Grado):\n", df_entities[['id', 'sub_type', 'degree']].head(3))
print(f"\nEsempio Eventi arricchiti (Primi 3 eventi con Ora):\n", df_events[['id', 'sub_type', 'hour', 'date_only']].head(3))

Inizio Fase 2.3: Arricchimento dati per la Visual Analytics...
-> Estrazione delle feature temporali dagli eventi...
-> Calcolo della Degree Centrality per le entità...

[SUCCESSO] Feature Engineering completata!
Esempio Entità arricchite (Primi 3 nodi con Grado):
             id sub_type  degree
0          Sam   Person      17
1        Kelly   Person       5
2  Nadia Conti   Person      37

Esempio Eventi arricchiti (Primi 3 eventi con Ora):
                     id    sub_type  hour   date_only
0   Event_Monitoring_0  Monitoring     8  2040-10-01
1  Event_Monitoring_33  Monitoring    10  2040-10-01
2  Event_Monitoring_38  Monitoring    -1     Unknown


In [18]:
import pandas as pd

print("Inizio Fase 2.3: Data Cleaning Avanzato")

# 1. Carichiamo i dataset arricchiti dal passo precedente
df_entities = pd.read_csv('../data/processed/nodes_entities.csv')
df_events = pd.read_csv('../data/processed/nodes_events.csv')
df_edges = pd.read_csv('../data/processed/edges_complete.csv')

# --- 2. PULIZIA NODI EVENTO (La parte più pesante del file) ---
# Eliminiamo le colonne che hanno oltre il 99% di valori mancanti o che sono ridondanti
colonne_da_cancellare_events = [
    'activity_type', 'participants', 'thing_collected', 'reference', 'date', 'time'
]

print(f"-> Rimozione colonne irrilevanti/ridondanti dagli Eventi: {colonne_da_cancellare_events}")
df_events_cleaned = df_events.drop(columns=[col for col in colonne_da_cancellare_events if col in df_events.columns])

# Pulizia del testo (Trimming): rimuoviamo spazi bianchi inutili all'inizio e alla fine dei messaggi
if 'content' in df_events_cleaned.columns:
    df_events_cleaned['content'] = df_events_cleaned['content'].astype(str).str.strip()


# --- 3. PULIZIA ARCHI (EDGES) ---
# Per evitare che D3.js fatichi a renderizzare linee sovrapposte inutili, 
# eliminiamo eventuali archi duplicati (stesso mittente, destinatario e ID)
edges_before = len(df_edges)
df_edges_cleaned = df_edges.drop_duplicates(subset=['source', 'target', 'id'])
print(f"-> Rimozione duplicati negli archi: eliminati {edges_before - len(df_edges_cleaned)} archi ridondanti.")


# --- 4. REPORT DI ALLEGGERIMENTO FILE ---
print("\n=== REPORT ALLEGGERIMENTO FILE (STILE ASCIONE) ===")
print(f"• Nodi Entità: {df_entities.shape[0]} righe | {df_entities.shape[1]} colonne")
print(f"• Nodi Evento - Colonne prima: {df_events.shape[1]} | Colonne dopo: {df_events_cleaned.shape[1]}")
print(f"• Archi totali ottimizzati e pronti: {len(df_edges_cleaned)}")


# --- 5. SALVATAGGIO DEI FILE DEFINITIVI ---
# Sovrascriviamo i file CSV finali nella cartella processed
df_events_cleaned.to_csv('../data/processed/nodes_events.csv', index=False)
df_edges_cleaned.to_csv('../data/processed/edges_complete.csv', index=False)

print("\n[SUCCESSO] I file sono stati alleggeriti e ottimizzati per il caricamento web!")

Inizio Fase 2.3: Data Cleaning Avanzato
-> Rimozione colonne irrilevanti/ridondanti dagli Eventi: ['activity_type', 'participants', 'thing_collected', 'reference', 'date', 'time']
-> Rimozione duplicati negli archi: eliminati 0 archi ridondanti.

=== REPORT ALLEGGERIMENTO FILE (STILE ASCIONE) ===
• Nodi Entità: 72 righe | 6 colonne
• Nodi Evento - Colonne prima: 23 | Colonne dopo: 17
• Archi totali ottimizzati e pronti: 3226

[SUCCESSO] I file sono stati alleggeriti e ottimizzati per il caricamento web!


## Clustering

In [20]:
import pandas as pd
import networkx as nx
# Importiamo l'algoritmo di Louvain integrato nativamente in NetworkX
from networkx.algorithms.community import louvain_communities

print("Inizio Fase 2.4: Calcoli Algoritmici Offline (Fasce Orarie e Louvain Nativo)...")

# 1. Carichiamo i dati puliti
df_entities = pd.read_csv('../data/processed/nodes_entities.csv')
df_events = pd.read_csv('../data/processed/nodes_events.csv')
df_edges = pd.read_csv('../data/processed/edges_complete.csv')


# --- PARTE 1: RAGGRUPPAMENTO PER FASCE ORARIE (TEMPORAL PATTERNS) ---
print("-> Generazione delle fasce orarie per gli eventi...")

def assegna_fascia_oraria(hour):
    if pd.isna(hour) or hour == -1:
        return 'Sconosciuto'
    elif 0 <= hour <= 5:
        return 'Notte (00:00 - 05:59)'
    elif 6 <= hour <= 11:
        return 'Mattina (06:00 - 11:59)'
    elif 12 <= hour <= 17:
        return 'Pomeriggio (12:00 - 17:59)'
    else:
        return 'Sera (18:00 - 23:59)'

df_events['time_slot'] = df_events['hour'].apply(assegna_fascia_oraria)
print(df_events['time_slot'].value_counts())


# --- PARTE 2: ALGORITMO DI PARTIZIONAMENTO (LOUVAIN NATIVO) ---
print("\n-> Esecuzione algoritmo di Louvain nativo per la Community Detection...")

# 2a. Creazione del grafo non orientato
G = nx.from_pandas_edgelist(df_edges, 'source', 'target', create_using=nx.Graph())

# 2b. Calcolo delle community con la funzione nativa di NetworkX
# Restituisce una lista di set, dove ogni set contiene i nodi di una community
communities_list = louvain_communities(G, seed=42) # seed fisso per avere sempre gli stessi risultati

# 2c. Trasformiamo la lista di set in un dizionario {Nodo: ID_Community} per mapparlo in Pandas
partition = {}
for community_id, node_set in enumerate(communities_list):
    for node in node_set:
        partition[node] = community_id

# 2d. Mappiamo i risultati sul nostro DataFrame delle Entità
df_entities['louvain_community'] = df_entities['id'].map(partition).fillna(-1).astype(int)

num_communities = len(communities_list)
print(f"L'algoritmo di Louvain ha identificato {num_communities} community distinte (fazioni).")
print("\nDistribuzione delle Entità nelle prime 5 Community più grandi:")
print(df_entities['louvain_community'].value_counts().head(5))


# --- PARTE 3: SALVATAGGIO DEFINITIVO ---
df_entities.to_csv('../data/processed/nodes_entities.csv', index=False)
df_events.to_csv('../data/processed/nodes_events.csv', index=False)

print("\n[SUCCESSO] Calcoli algoritmici offline completati e salvati con successo!")

Inizio Fase 2.4: Calcoli Algoritmici Offline (Fasce Orarie e Louvain Nativo)...
-> Generazione delle fasce orarie per gli eventi...
time_slot
Mattina (06:00 - 11:59)       599
Pomeriggio (12:00 - 17:59)    145
Sconosciuto                    32
Notte (00:00 - 05:59)          17
Sera (18:00 - 23:59)            9
Name: count, dtype: int64

-> Esecuzione algoritmo di Louvain nativo per la Community Detection...
L'algoritmo di Louvain ha identificato 11 community distinte (fazioni).

Distribuzione delle Entità nelle prime 5 Community più grandi:
louvain_community
8    10
3     9
5     8
6     8
2     7
Name: count, dtype: int64

[SUCCESSO] Calcoli algoritmici offline completati e salvati con successo!


## Salvataggio per D3

In [21]:
import pandas as pd
import json
import os

print("Inizio Fase 2.5: Esportazione del JSON Leggero per D3.js...")

# 1. Carichiamo i CSV finali puliti e arricchiti
df_entities = pd.read_csv('../data/processed/nodes_entities.csv')
df_events = pd.read_csv('../data/processed/nodes_events.csv')
df_relationships = pd.read_csv('../data/processed/nodes_relationships.csv')
df_edges = pd.read_csv('../data/processed/edges_complete.csv')

# 2. Gestione dei Valori Nulli (JSON/JavaScript non amano i NaN di Pandas)
# Sostituiamo gli eventuali NaN rimasti con stringhe vuote
df_entities = df_entities.fillna("")
df_events = df_events.fillna("")
df_relationships = df_relationships.fillna("")
df_edges = df_edges.fillna("")

# 3. Creazione della lista unica dei NODI
# Convertiamo le righe dei DataFrame in dizionari Python e li uniamo
print("-> Assemblaggio dell'array 'nodes'...")
nodes_list = []
nodes_list.extend(df_entities.to_dict(orient='records'))
nodes_list.extend(df_events.to_dict(orient='records'))
nodes_list.extend(df_relationships.to_dict(orient='records'))

# 4. Creazione della lista degli ARCHI
# In D3.js la convenzione standard è chiamarli 'links' anziché 'edges'
print("-> Assemblaggio dell'array 'links'...")
links_list = df_edges.to_dict(orient='records')

# 5. Strutturazione del dizionario finale
d3_dataset = {
    "nodes": nodes_list,
    "links": links_list
}

# 6. SALVATAGGIO OTTIMIZZATO (Minificazione)
output_path = '../data/processed/vast_mc3_optimized.json'
print("-> Compressione e salvataggio su disco...")

with open(output_path, 'w', encoding='utf-8') as f:
    # L'argomento separators=(',', ':') è il segreto: rimuove tutti gli spazi bianchi inutili
    # rendendo il file JSON una singola riga di testo iper-compressa.
    json.dump(d3_dataset, f, ensure_ascii=False, separators=(',', ':'))

# 7. Verifica delle performance (Peso del file)
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)

print("\n==================================================")
print(" [SUCCESSO] FASE 2 COMPLETATA MAGISTRALMENTE! ")
print("==================================================")
print(f"File esportato in: {output_path}")
print(f"-> Nodi esportati: {len(nodes_list)}")
print(f"-> Archi (links) esportati: {len(links_list)}")
print(f"-> Peso finale del payload web: {file_size_mb:.2f} MB")
print("\nIl dataset è ora leggerissimo, strutturato e pronto per non far crashare D3.js!")

Inizio Fase 2.5: Esportazione del JSON Leggero per D3.js...
-> Assemblaggio dell'array 'nodes'...
-> Assemblaggio dell'array 'links'...
-> Compressione e salvataggio su disco...

 [SUCCESSO] FASE 2 COMPLETATA MAGISTRALMENTE! 
File esportato in: ../data/processed/vast_mc3_optimized.json
-> Nodi esportati: 1159
-> Archi (links) esportati: 3226
-> Peso finale del payload web: 0.91 MB

Il dataset è ora leggerissimo, strutturato e pronto per non far crashare D3.js!
